# Tree accuracy: does the tree agree with the database taxonomy?

One notebook for **all tree variants**, so they can be compared on the same page. It is executed once by the
`tree_accuracy` rule after every tree, PhyloRank run and `assign_taxonomy` has finished. The question: **how well do
the database clade names (phylum ... genus) match the tree, and where they do not, is it a tree problem or a label
problem?** The alignment side (occupancy, divergence, genes) is in `tree_diagnostics`.

Variants are labelled `derep/copies`:

| part | values | meaning |
|---|---|---|
| derep | `all` / `derep` | every genome, or only the dereplicated representatives (external genomes always kept) |
| copies | `single` / `multi` | Complete BUSCO genes only, or also the top-scoring copy of Duplicated genes |

A quick-check subset run (label printed below) uses far fewer genomes: do not compare its numbers with a full run.
Trees with fewer genomes have fewer chances to fragment, so a drop in fragmentation between `all` and `derep` is not
by itself evidence of a better tree; the cleaner comparison is `single` vs `multi` on the same genomes.

Sections: (1) clade fragmentation, (2) support of the splits that cause it, (3) tips that look misplaced,
(4) RED, (5) how much the variants differ. Monophyly is **independent of where the tree is rooted**
(`scripts/tree_accuracy_lib.py`). Almost everything is a figure; the numbers are also written as TSV.

## 0. Setup

Parameters come from the JSON file named by `NB_PARAMS`, written by the Snakemake rule.

In [ ]:
import json
import os
import sys
from pathlib import Path

import numpy as np
import polars as pl
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
%matplotlib inline
from Bio import Phylo

In [ ]:
PARAMS = json.load(open(os.environ["NB_PARAMS"]))
sys.path.insert(0, PARAMS["scripts_dir"])
import tree_accuracy_lib as lib

OUT = Path(PARAMS["out_dir"])        # tables and PNG copies of the figures
OUT.mkdir(parents=True, exist_ok=True)
RANKS = lib.RANKS                    # phylum ... species
VARIANTS = [v for v in PARAMS["variants"] if Path(v["tree"]).exists()]
LABELS = [v["label"] for v in VARIANTS]
COLOURS = dict(zip(LABELS, plt.cm.tab10.colors))   # one colour per variant in every figure

print("Run     :", PARAMS["label"])
print("Variants:", ", ".join(LABELS))

In [ ]:
def show_all(df, width=200):
    '''
    Prints an entire polars dataframe in the notebook output.

    Parameters
    ----------
    df : pl.DataFrame
        The dataframe to be printed.
    width : int, optional
        The width (in characters) of the printed dataframe.
        Defaults to 200.
    '''
    with pl.Config() as cfg:
        cfg.set_tbl_cols(-1)
        cfg.set_tbl_rows(-1)
        cfg.set_tbl_width_chars(width)
        cfg.set_fmt_str_lengths(width)
        print(df)


def finish(fig, name):
    '''
    Saves a figure as PNG next to the tables and shows it in the notebook.

    Parameters
    ----------
    fig : matplotlib.figure.Figure
        The figure to save.
    name : str
        File name without extension.
    '''
    fig.tight_layout()
    fig.savefig(OUT / f"{name}.png", dpi=130, bbox_inches="tight")
    plt.show()


def grouped_bars(axis, table, categories, title, ylabel=""):
    '''
    Draws one group of bars per category with one bar per variant.

    Parameters
    ----------
    axis : matplotlib.axes.Axes
        Axis to draw on.
    table : dict
        {variant label: list of values, one per category}.
    categories : list of str
        Category names (x tick labels).
    title, ylabel : str
        Axis title and y label.
    '''
    width = 0.8 / max(1, len(table))
    for k, label in enumerate(table):
        axis.bar(np.arange(len(categories)) + k * width, table[label], width, color=COLOURS[label], label=label)
    axis.set_xticks(np.arange(len(categories)) + 0.4 - width / 2)
    axis.set_xticklabels(categories)
    axis.set_title(title)
    axis.set_ylabel(ylabel)

## 1. Load the trees and find the clade pieces

If a database name (say an order) is monophyletic, all its tips form **one piece**: one part of the tree that
holds tips of that name and no labelled tips of any other name. Tips of the same name elsewhere form **extra
pieces**. Unlabelled tips (e.g. MAGs) are ignored, so they never count against a name.

In [ ]:
taxonomy = lib.read_taxonomy(PARAMS["taxonomy"])   # {genome: {rank: name}} database lineages

TREES, PIECES, SUMMARY, NAMES = {}, {}, {}, {}
for variant in VARIANTS:
    label = variant["label"]
    TREES[label] = Phylo.read(variant["tree"], "newick")
    PIECES[label] = {rank: lib.pieces_unrooted(TREES[label], taxonomy, rank) for rank in RANKS}
    SUMMARY[label], NAMES[label] = lib.fragmentation(TREES[label], taxonomy, pieces_by_rank=PIECES[label])
    n_tips = TREES[label].count_terminals()
    n_labelled = sum(1 for tip in TREES[label].get_terminals() if tip.name in taxonomy)
    print(f"{label:14} {n_tips} tips, {n_labelled} with a database taxonomy")

## 2. Clade fragmentation

Per rank: how many extra pieces there are, how many names are split, and how many tips sit in split names.

**How to read it.**

* A few extra pieces at deep ranks (phylum, class) are usually real: NCBI groups such as Pichiomycetes or Bigyra are
  known to be non-monophyletic.
* Fragmentation that grows toward the tips (many extra pieces at order and family) is more likely wrong labels,
  contaminated genomes or an unstable tree.
* The heatmap lists the most fragmented names with their number of pieces in every variant, so you can see which
  clades a genome set or gene set changes.

In [ ]:
fragmentation = pl.concat([pl.DataFrame(SUMMARY[label]).with_columns(pl.lit(label).alias("variant")) for label in LABELS])
fragmentation.write_csv(OUT / "fragmentation.tsv", separator="\t")
by_rank = lambda col: {label: fragmentation.filter(pl.col("variant") == label).sort(
    pl.col("rank").replace_strict(RANKS, list(range(len(RANKS)))))[col].to_list() for label in LABELS}

fig, axes = plt.subplots(1, 3, figsize=(17, 3.8))
grouped_bars(axes[0], by_rank("extra_pieces"), RANKS, "extra pieces (beyond one per name)")
grouped_bars(axes[1], by_rank("fragmented_names"), RANKS, "names split into several pieces")
grouped_bars(axes[2], by_rank("tips_in_fragmented"), RANKS, "tips in split names")
axes[0].legend(fontsize=8)
finish(fig, "fragmentation_by_rank")

In [ ]:
N_SHOWN = 25
pieces_table = (pl.concat([pl.DataFrame(NAMES[label]).select("rank", "name", "tips", "pieces")
                           .with_columns(pl.lit(label).alias("variant")) for label in LABELS])
                .pivot(on="variant", index=["rank", "name"], values="pieces").fill_null(0))
pieces_table = pieces_table.with_columns(
    pl.max_horizontal(LABELS).alias("max_pieces"),
    (pl.max_horizontal(LABELS) - pl.min_horizontal(LABELS)).alias("spread"))
pieces_table.write_csv(OUT / "pieces_by_name.tsv", separator="\t")

top = pieces_table.filter(pl.col("rank").is_in(["class", "order", "family"])).sort("max_pieces", descending=True).head(N_SHOWN)
grid = top.select(LABELS).to_numpy()
fig, axis = plt.subplots(figsize=(1.0 * len(LABELS) + 4, 0.28 * N_SHOWN + 1.5))
image = axis.imshow(grid, aspect="auto", cmap="magma_r")
axis.set_xticks(range(len(LABELS)))
axis.set_xticklabels(LABELS, rotation=45, ha="right")
axis.set_yticks(range(top.height))
axis.set_yticklabels([f"{r[0]}  {n}" for r, n in zip(top["rank"], top["name"])], fontsize=8)
for i in range(grid.shape[0]):
    for j in range(grid.shape[1]):
        axis.text(j, i, int(grid[i, j]), ha="center", va="center", fontsize=7,
                  color="white" if grid[i, j] > grid.max() / 2 else "black")
plt.colorbar(image, label="pieces")
axis.set_title(f"{N_SHOWN} most fragmented names (class/order/family): pieces per variant")
finish(fig, "most_fragmented_names")

## 3. Are the splits that fragment a clade supported?

Each extra piece hangs below a branch. Its **support** is FastTree's SH-like value on that branch (0-1).

* **high (>= 0.95)** - the data confidently place those tips elsewhere. The disagreement with the database is real
  (a wrong label, a contaminated genome or biology) and the tree is not the problem.
* **low** - the placement is uncertain and the tree may simply be wrong there.

SH-like values are generous, so read high support as "not obviously noise", not as proof.

In [ ]:
support_rows = [{"variant": label, "rank": rank, "name": name, "piece_tips": size, "support": support}
                for label in LABELS for rank in RANKS
                for name, size, support in lib.extra_piece_supports(PIECES[label][rank])]
extra_support = pl.DataFrame(support_rows, schema={"variant": pl.Utf8, "rank": pl.Utf8, "name": pl.Utf8,
                                                    "piece_tips": pl.Int64, "support": pl.Float64})
extra_support.write_csv(OUT / "extra_piece_support.tsv", separator="\t")
with_support = extra_support.drop_nulls("support")

if with_support.height:
    fig, axes = plt.subplots(1, 3, figsize=(17, 3.8))
    for label in LABELS:
        sub = with_support.filter(pl.col("variant") == label)
        axes[0].hist(sub["support"], bins=20, range=(0, 1), histtype="step", linewidth=1.6, color=COLOURS[label], label=label)
        axes[1].scatter(sub["piece_tips"], sub["support"], s=8, alpha=0.5, color=COLOURS[label])
    axes[0].set_title("support of the branch above each extra piece")
    axes[0].legend(fontsize=8)
    axes[1].set_xscale("log")
    axes[1].set_xlabel("piece size (tips)")
    axes[1].set_ylabel("support")
    shares = {label: [(with_support.filter(pl.col("variant") == label)["support"] < 0.8).mean(),
                      (with_support.filter(pl.col("variant") == label)["support"] >= 0.95).mean()] for label in LABELS}
    grouped_bars(axes[2], shares, ["support < 0.8", "support >= 0.95"], "share of extra pieces")
    finish(fig, "extra_piece_support")
else:
    print("No support values in these trees.")

## 4. Tips that look misplaced

A tip is flagged at a rank when its nearest labelled neighbours (at least 8) **agree at >= 90% on a different name**.
`sister = True` means that name belongs to the same parent clade (for example two orders of one class), a milder
disagreement that is often just a classification difference. The counts below separate the two; the genome checks
use only the non-sister tips.

Flagged genomes are then compared with all genomes on BUSCO completeness, BUSCO duplication (a high fraction can mean
contamination or a mixed assembly) and whether their database taxonomy is only `partial`.

**How to read it.** If flagged genomes look like everyone else, the cause is most likely the label (or real
biology). If they are less complete, more duplicated or more often `partial`, look at the genomes first.

In [ ]:
RANKS_FLAGGED = ["phylum", "class", "order", "family"]
flagged = {label: {rank: lib.misplaced_tips(TREES[label], taxonomy, rank) for rank in RANKS_FLAGGED} for label in LABELS}

misplaced = pl.DataFrame(
    [{"variant": label, "rank": rank, **row} for label in LABELS for rank in RANKS_FLAGGED for row in flagged[label][rank]],
    schema={"variant": pl.Utf8, "rank": pl.Utf8, "tip": pl.Utf8, "label": pl.Utf8, "neighbours_say": pl.Utf8,
            "agreement": pl.Float64, "n_neighbours": pl.Int64, "sister": pl.Boolean})
misplaced.write_csv(OUT / "misplaced_tips.tsv", separator="\t")

fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
grouped_bars(axes[0], {l: [sum(not r["sister"] for r in flagged[l][k]) for k in RANKS_FLAGGED] for l in LABELS},
             RANKS_FLAGGED, "non-sister misplaced tips", "tips")
grouped_bars(axes[1], {l: [sum(r["sister"] for r in flagged[l][k]) for k in RANKS_FLAGGED] for l in LABELS},
             RANKS_FLAGGED, "sister-group swaps", "tips")
axes[0].legend(fontsize=8)
finish(fig, "misplaced_counts")

In [ ]:
# tax_status (complete / partial / unresolved) and the BUSCO full table of every genome
mapping = pl.read_csv(PARAMS["taxonomy_mapping"], separator="\t").select("acc", "tax_status")
tax_status = dict(zip(mapping["acc"], mapping["tax_status"]))
busco_tables = {}
for entry in PARAMS["busco_manifests"]:
    manifest, base = Path(entry["manifest"]), Path(entry["base"])
    if manifest.exists():
        qc = pl.read_csv(manifest, separator="\t")
        busco_tables.update({a: Path(r) if Path(r).is_absolute() else base / r
                             for a, r in zip(qc["genome_acc"], qc["busco_result"])})

_cache = {}
def busco_stats(accession):
    '''
    BUSCO completeness and duplication of one genome (cached across variants).

    Parameters
    ----------
    accession : str
        Genome accession (tree tip name).

    Returns
    -------
    tuple or None
        (Complete count, Duplicated count, Duplicated fraction), or None without a BUSCO table.
    '''
    if accession not in _cache:
        path = busco_tables.get(accession)
        if path is None or not Path(path).exists():
            _cache[accession] = None
        else:
            counts = lib.busco_status_counts(path)
            total = sum(counts.values()) or 1
            _cache[accession] = (counts.get("Complete", 0), counts.get("Duplicated", 0), counts.get("Duplicated", 0) / total)
    return _cache[accession]

In [ ]:
def group_values(accessions):
    '''Lists of completeness, duplication fraction and partial-taxonomy flags for a set of genomes.'''
    stats = [s for s in map(busco_stats, accessions) if s]
    return ([s[0] for s in stats], [s[2] for s in stats],
            [tax_status.get(a) == "partial" for a in accessions])


fig, axes = plt.subplots(1, 3, figsize=(17, 4))
positions, data_c, data_d, ticks, tick_pos, partial = [], [], [], [], [], []
for k, label in enumerate(LABELS):
    odd = sorted({r["tip"] for rank in ("order", "family") for r in flagged[label][rank] if not r["sister"]})
    everyone = [t.name for t in TREES[label].get_terminals()]
    for j, (group, name) in enumerate(((odd, "misplaced"), (everyone, "all tips"))):
        c, d, p = group_values(group)
        pos = k * 3 + j
        positions.append(pos); data_c.append(c or [np.nan]); data_d.append(d or [np.nan])
        ticks.append(f"{label}\n{name} ({len(group)})"); tick_pos.append(pos)
        partial.append(float(np.mean(p)) if p else 0.0)
colours = [COLOURS[LABELS[p // 3]] if p % 3 == 0 else "lightgrey" for p in positions]
for axis, data, title in ((axes[0], data_c, "BUSCO Complete genes"), (axes[1], data_d, "BUSCO Duplicated fraction")):
    box = axis.boxplot(data, positions=positions, widths=0.8, patch_artist=True, showfliers=False)
    for patch, colour in zip(box["boxes"], colours):
        patch.set_facecolor(colour)
    axis.set_xticks(tick_pos); axis.set_xticklabels(ticks, rotation=60, ha="right", fontsize=7)
    axis.set_title(title)
axes[2].bar(positions, partial, color=colours)
axes[2].set_xticks(tick_pos); axes[2].set_xticklabels(ticks, rotation=60, ha="right", fontsize=7)
axes[2].set_title("share with only a partial database taxonomy")
finish(fig, "misplaced_genome_quality")

Tips flagged as non-sister misplaced at order or family level in the most variants. A tip flagged in every
variant is a robust problem (a label, contamination, or real biology); one that appears in a single variant is
sensitive to the genome or gene set.

In [ ]:
N_TIPS = 25
flag_counts = (misplaced.filter(~pl.col("sister") & pl.col("rank").is_in(["order", "family"]))
               .group_by("tip").agg(pl.col("variant").n_unique().alias("variants"),
                                    pl.col("label").first(), pl.col("neighbours_say").first()))
shown = flag_counts.sort(["variants", "tip"], descending=[True, False]).head(N_TIPS)
if shown.height:
    grid = np.array([[any(r["tip"] == tip and not r["sister"] for rank in ("order", "family")
                          for r in flagged[l][rank]) for l in LABELS] for tip in shown["tip"]], dtype=float)
    fig, axis = plt.subplots(figsize=(1.0 * len(LABELS) + 5, 0.28 * shown.height + 1.5))
    axis.imshow(grid, aspect="auto", cmap="Reds", vmin=0, vmax=1)
    axis.set_xticks(range(len(LABELS)))
    axis.set_xticklabels(LABELS, rotation=45, ha="right")
    axis.set_yticks(range(shown.height))
    axis.set_yticklabels([f"{t}  {a} -> {b}" for t, a, b in zip(shown["tip"], shown["label"], shown["neighbours_say"])], fontsize=7)
    axis.set_title(f"top {shown.height} non-sister misplaced tips (order/family); red = flagged")
    finish(fig, "misplaced_tips_by_variant")
else:
    print("No non-sister misplaced tips.")

## 5. RED (relative evolutionary divergence)

PhyloRank gives every node a RED value between 0 (root) and 1 (tips); `assign_taxonomy` uses RED cutoffs per rank to
decide how deep a database name may be inherited by the tips below a node. The cutoffs are the median RED of the
existing names at each rank **on that tree**, so they differ between variants.

**How to read it.**

* *Spread*: boxes are the RED of each named taxon at a rank, the red mark is the cutoff. If boxes of neighbouring
  ranks overlap a lot, one cutoff cannot separate the ranks. The `wrong side` panel gives the share of names whose
  RED lies beyond the neighbouring rank's median: small is good; 10-20% means RED only roughly follows the database
  ranks (expected, as NCBI ranks are not equal in age across eukaryotes).
* *Monotonicity*: PhyloRank's RED is a median over several rootings, so a child can end up below its parent, which
  also gives negative branch lengths in the RED tree. `assign_taxonomy` ignores those lengths and raises each node's
  RED to at least its parent's. A handful of nodes is fine; many would mean the tree is poorly rooted or unstable near
  the root.

In [ ]:
RED_RANKS = ["phylum", "class", "order", "family", "genus"]
red = {v["label"]: lib.red_by_rank(v["red_tsv"], v["red_dict"]) for v in VARIANTS}

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
width = 0.8 / len(LABELS)
for k, label in enumerate(LABELS):
    for i, rank in enumerate(RED_RANKS):
        cutoff, values = red[label].get(rank, (None, []))
        if not values:
            continue
        pos = i + k * width
        box = axes[0].boxplot([values], positions=[pos], widths=width * 0.85, patch_artist=True, showfliers=False)
        box["boxes"][0].set_facecolor(COLOURS[label])
        axes[0].plot([pos - width * 0.4, pos + width * 0.4], [cutoff, cutoff], "r-", linewidth=2)
axes[0].set_xticks(np.arange(len(RED_RANKS)) + 0.4 - width / 2)
axes[0].set_xticklabels(RED_RANKS)
axes[0].set_title("RED of each named taxon (red mark = cutoff)")
from matplotlib.patches import Patch
axes[0].legend(handles=[Patch(color=COLOURS[l], label=l) for l in LABELS], fontsize=8)
overlap = {label: lib.rank_overlap(red[label]) for label in LABELS}
grouped_bars(axes[1], {l: [np.nanmean(overlap[l].get(r, (np.nan, np.nan))) for r in RED_RANKS] for l in LABELS},
             RED_RANKS, "share of names on the wrong side of the neighbouring rank")
rows = []
for variant in VARIANTS:
    n_nodes, n_below, n_negative, worst = lib.red_monotonicity(Phylo.read(variant["red_tree"], "newick"))
    rows.append({"variant": variant["label"], "internal_nodes": n_nodes, "nodes_below_parent": n_below,
                 "negative_branches": n_negative, "most_negative": worst})
monotonicity = pl.DataFrame(rows)
x = np.arange(len(LABELS))
axes[2].bar(x - 0.2, monotonicity["nodes_below_parent"], 0.4, label="nodes below their parent")
axes[2].bar(x + 0.2, monotonicity["negative_branches"], 0.4, label="negative branches")
axes[2].set_xticks(x); axes[2].set_xticklabels(LABELS, rotation=30, ha="right")
axes[2].set_title("non-monotone RED in PhyloRank's tree")
axes[2].legend(fontsize=8)
finish(fig, "red")

monotonicity.write_csv(OUT / "red_monotonicity.tsv", separator="\t")
pl.DataFrame([{"variant": l, "rank": r, "cutoff": red[l][r][0], "taxa": len(red[l][r][1]),
               "median": float(np.median(red[l][r][1])) if red[l][r][1] else None}
              for l in LABELS for r in red[l]]).write_csv(OUT / "red_summary.tsv", separator="\t")

## 6. How different are the variants?

Normalised **Robinson-Foulds** distance on the tips two trees share: the share of branches found in one tree but not
the other (0 = identical topology, 1 = nothing in common).

**How to read it.** Compare the distance caused by one choice with the distance caused by another. If `single` vs
`multi` (same genomes) is much smaller than `all` vs `derep` (different genomes), the Duplicated genes barely change
the tree and the genome set matters more. A few percent is what tree-search noise alone gives, so only large
differences are meaningful.

In [ ]:
rf = np.zeros((len(LABELS), len(LABELS)))
for i, first in enumerate(LABELS):
    for j, second in enumerate(LABELS):
        if i < j:
            rf[i, j] = rf[j, i] = lib.rf_distance(TREES[first], TREES[second])[1]
pl.DataFrame({"tree": LABELS, **{l: np.round(rf[:, k], 4) for k, l in enumerate(LABELS)}}).write_csv(
    OUT / "rf_matrix.tsv", separator="\t")

fig, axis = plt.subplots(figsize=(1.0 * len(LABELS) + 3, 1.0 * len(LABELS) + 2))
image = axis.imshow(rf, vmin=0, cmap="viridis")
axis.set_xticks(range(len(LABELS))); axis.set_xticklabels(LABELS, rotation=45, ha="right")
axis.set_yticks(range(len(LABELS))); axis.set_yticklabels(LABELS)
for i in range(len(LABELS)):
    for j in range(len(LABELS)):
        axis.text(j, i, f"{rf[i, j]:.3f}", ha="center", va="center", color="white", fontsize=8)
plt.colorbar(image, label="normalised Robinson-Foulds distance")
axis.set_title("topological distance between variants")
finish(fig, "rf_matrix")

## 7. Summary

One row per variant. Tables and PNG copies of every figure are written next to the executed notebook
(`tables/tree_accuracy/`).

In [ ]:
summary = pl.DataFrame([{
    "variant": label,
    "tips": TREES[label].count_terminals(),
    "extra_pieces_order": [r for r in SUMMARY[label] if r["rank"] == "order"][0]["extra_pieces"],
    "extra_pieces_family": [r for r in SUMMARY[label] if r["rank"] == "family"][0]["extra_pieces"],
    "extra_pieces_support>=0.95": round(float((with_support.filter(pl.col("variant") == label)["support"] >= 0.95).mean()), 3)
                                  if with_support.height else None,
    "non_sister_misplaced_order": sum(not r["sister"] for r in flagged[label]["order"]),
    "red_nodes_below_parent": int(monotonicity.filter(pl.col("variant") == label)["nodes_below_parent"][0]),
} for label in LABELS])
show_all(summary)